# Prefix confidence-gating figure (fig:prefix-gated)

Extracted from `protonet_ablations.ipynb` (cells 47/49) so the paper figure
can be restyled without the ablations kernel. Reads cached
`prefix_results.json` only (no refresh logic; caches are current).
Gated frontier: accuracy of early-stopped rollouts vs % generation steps
saved, theta swept 0.55-0.95, points with <10% stop rate omitted.
AAAI: >=9pt at true include size (canvas = columnwidth), Times, Type-42.
Writes `paper/figures/fig_prefix_gated.pdf` (+ `_attn` companion).

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..")
FIGS = ROOT / "paper/figures"
PREFIX_RES = json.load(open(ROOT / "exp/results/protonet_ablations/prefix_results.json"))

PREFIX_SEED = 0
THETAS = [0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
MODELS = ["gpt5mini", "qwen", "deepseek"]
FIG_POOLS = {"tau2:g+q+d": "Tau2-Bench", "bfclv3:g+q+d": "BFCLv3",
             "agentclinic:g+q+d": "AgentClinic"}
BENCH_COLOR = {"tau2:g+q+d": "#0072B2", "bfclv3:g+q+d": "#E69F00",
               "agentclinic:g+q+d": "#009E73"}
ANNOT = (0.6, 0.7, 0.8, 0.9)
MIN_STOP = 0.10    # omit points where fewer than 10% of rollouts stop
ANNOT_OFFSET = {"tau2:g+q+d": (0, 8), "bfclv3:g+q+d": (11, 6),
                "agentclinic:g+q+d": (0, 7)}    # label offsets per curve (pts)
FIGSIZE = (3.3125, 2.1)   # AAAI columnwidth, include at width=\\columnwidth

PAPER_RC = {
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 11,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 9,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "figure.dpi": 250,
}
plt.rcParams.update(PAPER_RC)
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
})

def curve(pool, variant):
    """(xs, ys, keep-indices) for a pool's in-domain gated frontier."""
    res = PREFIX_RES[f"{pool}|{variant}|s{PREFIX_SEED}"]
    ds = pool.split(":")[0]
    cc = [f"{ds}/{m}" for m in MODELS]
    xs, ys, srs = [], [], []
    for th in THETAS:
        g = [res["cells"][c]["gated"][str(th)] for c in cc if c in res["cells"]]
        xs.append(100 * np.mean([x["steps_saved"] for x in g]))
        ys.append(np.nanmean([x["acc"] for x in g]))
        srs.append(np.mean([x["stop_rate"] for x in g]))
    keep = [i for i, sr in enumerate(srs) if sr >= MIN_STOP]
    return xs, ys, keep

In [ ]:
# ---- fig_prefix_gated: gated frontier, all benchmarks, one panel ----
# per-(pool, theta) offset overrides to de-collide labels; falls back to
# the pool default in ANNOT_OFFSET
ANNOT_OVERRIDE = {
    ("tau2:g+q+d", 0.8): (2, 9),         # above own point
    ("bfclv3:g+q+d", 0.8): (-18, -7),    # below-left, clear of the 0.85 dot
    ("bfclv3:g+q+d", 0.6): (-16, -5),    # left of point, clear of AC 0.6
    ("tau2:g+q+d", 0.6): (8, 6),
}
fig, ax = plt.subplots(figsize=FIGSIZE)
for pool, label in FIG_POOLS.items():
    xs, ys, keep = curve(pool, "uniform")
    ax.plot([xs[i] for i in keep], [ys[i] for i in keep],
            marker="o", ms=3.5, lw=1.6, color=BENCH_COLOR[pool], label=label)
    for th in ANNOT:
        i = THETAS.index(th)
        if i in keep:
            dx, dy = ANNOT_OVERRIDE.get((pool, th), ANNOT_OFFSET[pool])
            ax.annotate(rf"$\theta$={th}", (xs[i], ys[i]), xytext=(dx, dy),
                        textcoords="offset points", fontsize=9,
                        color=BENCH_COLOR[pool], ha="center",
                        bbox=dict(boxstyle="round,pad=0.1", fc="white",
                                  ec="none", alpha=0.75))
ax.set_xlabel("Steps saved (%)")
ax.set_ylabel("Early-stopped accuracy")
ax.legend(frameon=False, loc="best", handlelength=1.6,
          labelspacing=0.3, handletextpad=0.5)
fig.subplots_adjust(left=0.16, right=0.985, top=0.97, bottom=0.21)
FIGS.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGS / "fig_prefix_gated.pdf")
fig.savefig(FIGS / "fig_prefix_gated.png", dpi=200)
print("saved", FIGS / "fig_prefix_gated.{pdf,png}")

In [ ]:
# ---- fig_prefix_gated_attn: uniform (solid) vs uniform_attn (dashed) ----
import matplotlib.lines as mlines
CMP_VARIANTS = {"uniform": "-", "uniform_attn": "--"}
fig, ax = plt.subplots(figsize=(3.3125, 2.6))
for pool, label in FIG_POOLS.items():
    for variant, ls in CMP_VARIANTS.items():
        xs, ys, keep = curve(pool, variant)
        ax.plot([xs[i] for i in keep], [ys[i] for i in keep], ls,
                marker="o", ms=3, lw=1.5, color=BENCH_COLOR[pool])
ax.set_xlabel("Steps saved (%)")
ax.set_ylabel("Early-stopped accuracy")
handles = ([mlines.Line2D([], [], color=BENCH_COLOR[p], lw=1.6, label=l)
            for p, l in FIG_POOLS.items()]
           + [mlines.Line2D([], [], color="#555555", lw=1.5, ls=ls,
                            label={"uniform": "mean pool",
                                   "uniform_attn": "attn pool"}[v])
              for v, ls in CMP_VARIANTS.items()])
ax.legend(handles=handles, frameon=False, ncol=1, fontsize=9,
          handlelength=1.6, labelspacing=0.25, handletextpad=0.5,
          loc="upper right")
fig.subplots_adjust(left=0.145, right=0.985, top=0.97, bottom=0.17)
fig.savefig(FIGS / "fig_prefix_gated_attn.pdf")
fig.savefig(FIGS / "fig_prefix_gated_attn.png", dpi=200)
print("saved", FIGS / "fig_prefix_gated_attn.{pdf,png}")